# Deep Learning for Image Analysis
### YOLO Object Detection on Pascal VOC 2012

**Authors:** Bo Fu, Yehoshua Perez Condori  
**Programme:** MSc Artificial Intelligence  
**Institution:** City St George's, University of London  
**Module:** INM705 Deep Learning for Image Analysis  
**Module Leader:** Dr Riad Ibadulla  

---

### Project Overview
Implementation of YOLOv1-style object detection using VGG16 backbone trained on Pascal VOC 2012 dataset with 20 object categories.

---

### Links
- **GitHub:** https://github.com/BoFu001/YOLO-object-detection
- **Kaggle Notebook:** https://www.kaggle.com/code/bofu001/yolo-object-detection
- **Kaggle Dataset:** https://www.kaggle.com/datasets/huanghanchina/pascal-voc-2012
- **Wandb:** https://wandb.ai/bofu001-city-st-george-s-university-of-london/YOLO-VOC2012

In [ ]:
import random
import numpy as np
import torch
from config import CKPT_DIR, DEVICE, SEED

# reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# custom modules
from modules.Dataset import get_dataloaders
from modules.Train import train
from modules.TrainFinetune import train_finetune
from modules.TrainFinetune2 import train_finetune2
from modules.Evaluation import evaluate
from modules.Models.YOLOv1 import YOLOv1
from modules.Models.YOLOv1Dropout import YOLOv1Dropout
from modules.Models.YOLOv1Finetune import YOLOv1Finetune

In [ ]:
# YOLO parameters
S = 7
B = 2
C = 20

# training parameters
BATCH_SIZE   = 16 
EPOCHS       = 5
LR           = 1e-3
WEIGHT_DECAY = 1e-4

# loss weights
LAMBDA_BOX   = 5.0
LAMBDA_NOOBJ = 0.5

# inference thresholds
CONF_THRESH    = 0.30
NMS_IOU_THRESH = 0.45

In [ ]:
# create all data loaders
train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE, S, B, C)

#### Experiment 1 - Baseline
* LR: 1e-3
* EPOCHS: 5
* Goal: verify model can learn, check loss trend

In [ ]:
# RUN_NAME = "exp1_lr1e-3_ep5"
# CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"

In [ ]:
# # create model
# model = YOLOv1(S=S, B=B, C=C).to(DEVICE)

# # multi-GPU support
# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs!")
#     model = torch.nn.DataParallel(model)

# # unwrap DataParallel
# raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

In [ ]:
# # training
# raw_model = train(
#     model=model,
#     raw_model=raw_model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     S=S, B=B, C=C,
#     BATCH_SIZE=BATCH_SIZE,
#     EPOCHS=EPOCHS,
#     LR=LR,
#     WEIGHT_DECAY=WEIGHT_DECAY,
#     LAMBDA_BOX=LAMBDA_BOX,
#     LAMBDA_NOOBJ=LAMBDA_NOOBJ,
#     RUN_NAME=RUN_NAME
# )

# # save trained weights
# torch.save(raw_model.state_dict(), CKPT_PATH)
# print(f"trained weights saved: {RUN_NAME}.pth")

In [ ]:
# # evaluation

# # load trained weights before evaluation
# raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

# results = evaluate(
#     model=model,
#     test_loader=test_loader,
#     S=S, B=B, C=C,
#     conf_thresh=CONF_THRESH,
#     iou_thresh=NMS_IOU_THRESH
# )

#### Experiment 2 - Lower Learning Rate
* LR: 1e-3 → 1e-4
* EPOCHS: 60
* Goal: reduce overfitting seen in Experiment 1

In [ ]:
RUN_NAME   = "exp2_lr1e-4_ep60"
LR         = 1e-4  
EPOCHS     = 60    
CKPT_PATH  = f"{CKPT_DIR}/{RUN_NAME}.pth"

In [ ]:
# # training
# raw_model = train(
#     model=model,
#     raw_model=raw_model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     S=S, B=B, C=C,
#     BATCH_SIZE=BATCH_SIZE,
#     EPOCHS=EPOCHS,
#     LR=LR,
#     WEIGHT_DECAY=WEIGHT_DECAY,
#     LAMBDA_BOX=LAMBDA_BOX,
#     LAMBDA_NOOBJ=LAMBDA_NOOBJ,
#     RUN_NAME=RUN_NAME
# )

# # save trained weights
# torch.save(raw_model.state_dict(), CKPT_PATH)
# print(f"trained weights saved: {RUN_NAME}.pth")

In [ ]:
# # evaluation

# # load trained weights before evaluation
# raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

# results = evaluate(
#     model=model,
#     test_loader=test_loader,
#     S=S, B=B, C=C,
#     conf_thresh=CONF_THRESH,
#     iou_thresh=NMS_IOU_THRESH
# )

#### Experiment 3 - Dropout
* LR: 1e-4
* EPOCHS: 60
* Dropout: 0.5 added in head of YOLOv1Dropout
* Goal: reduce overfitting

In [ ]:
RUN_NAME  = "exp3_dropout0.5_ep60"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"

In [ ]:
# create model
model = YOLOv1Dropout(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model


# training
raw_model = train(
    model=model, 
    raw_model=raw_model,
    train_loader=train_loader, 
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE, 
    EPOCHS=EPOCHS, LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY, 
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ, 
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model, 
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH, 
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 4 - Finetune VGG last 2 layers
* LR: 1e-4
* EPOCHS: 60
* Dropout: 0.5 in head of YOLOv1Finetune
* Unfreeze: last 2 conv layers of VGG16
* Goal: improve feature extraction for VOC dataset
#### Experiment 4b - Finetune with Early Stopping
* Same setup as exp4
* Early Stopping: patience=5, based on val loss
* Goal: save best model at lowest val loss, avoid overfitting seen in exp4

In [ ]:
RUN_NAME  = "exp4b_finetune_ep60"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"

In [ ]:
# # create model
# model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# # multi-GPU support
# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs!")
#     model = torch.nn.DataParallel(model)

# # unwrap DataParallel
# raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

# # training
# raw_model = train_finetune(
#     model=model, 
#     raw_model=raw_model,
#     train_loader=train_loader, 
#     val_loader=val_loader,
#     S=S, B=B, C=C,
#     BATCH_SIZE=BATCH_SIZE, 
#     EPOCHS=EPOCHS, 
#     LR=LR,
#     WEIGHT_DECAY=WEIGHT_DECAY, 
#     LAMBDA_BOX=LAMBDA_BOX,
#     LAMBDA_NOOBJ=LAMBDA_NOOBJ, 
#     RUN_NAME=RUN_NAME
# )
# # save trained weights
# torch.save(raw_model.state_dict(), CKPT_PATH)
# print(f"trained weights saved: {RUN_NAME}.pth")

# # evaluation
# raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
# results = evaluate(
#     model=model, 
#     test_loader=test_loader,
#     S=S, B=B, C=C,
#     conf_thresh=CONF_THRESH, 
#     iou_thresh=NMS_IOU_THRESH
# )

#### Experiment 5 - Layer-wise Learning Rate
* LR backbone: 1e-5
* LR head: 1e-4
* Early Stopping: patience=5
* Dropout: 0.5 in head of YOLOv1Finetune
* Unfreeze: last 2 conv layers of VGG16
* Goal: more stable fine-tuning of backbone

In [ ]:
RUN_NAME  = "exp5_layerwise_lr_ep60"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
LR_HEAD=1e-4
LR_BACKBONE=1e-5

In [ ]:
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

# training
raw_model = train_finetune2(
    model=model, 
    raw_model=raw_model,
    train_loader=train_loader, 
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE, 
    EPOCHS=EPOCHS, 
    LR_HEAD=LR_HEAD,
    LR_BACKBONE=LR_BACKBONE,
    WEIGHT_DECAY=WEIGHT_DECAY, 
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ, 
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model, 
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH, 
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 6 - Layer-wise LR tuning
* LR head: 1e-4
* LR backbone: 5e-5
* Early Stopping: patience=5
* Dropout: 0.5 in head of YOLOv1Finetune
* Unfreeze: last 2 conv layers of VGG16
* Goal: find better backbone LR

In [ ]:
RUN_NAME    = "exp6_layerwise_lr2"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
LR_BACKBONE = 5e-5 

In [ ]:
# # create model
# model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# # multi-GPU support
# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs!")
#     model = torch.nn.DataParallel(model)

# # unwrap DataParallel
# raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

# # training
# raw_model = train_finetune2(
#     model=model, 
#     raw_model=raw_model,
#     train_loader=train_loader, 
#     val_loader=val_loader,
#     S=S, B=B, C=C,
#     BATCH_SIZE=BATCH_SIZE, 
#     EPOCHS=EPOCHS, 
#     LR_HEAD=LR_HEAD,
#     LR_BACKBONE=LR_BACKBONE,
#     WEIGHT_DECAY=WEIGHT_DECAY, 
#     LAMBDA_BOX=LAMBDA_BOX,
#     LAMBDA_NOOBJ=LAMBDA_NOOBJ, 
#     RUN_NAME=RUN_NAME
# )
# # save trained weights
# torch.save(raw_model.state_dict(), CKPT_PATH)
# print(f"trained weights saved: {RUN_NAME}.pth")

# # evaluation
# raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
# results = evaluate(
#     model=model, 
#     test_loader=test_loader,
#     S=S, B=B, C=C,
#     conf_thresh=CONF_THRESH, 
#     iou_thresh=NMS_IOU_THRESH
# )

# Experiment 7 - Data Augmentation
* Same as exp4b
* Added ColorJitter augmentation on training set
* Goal: reduce overfitting with data augmentation

In [ ]:
RUN_NAME = "exp7_augmentation"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
LR = 1e-4

In [ ]:
# create data loaders with augmentation
train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE, S, B, C, augment=True)

In [ ]:
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

# training
raw_model = train_finetune(
    model=model, 
    raw_model=raw_model,
    train_loader=train_loader, 
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE, 
    EPOCHS=EPOCHS, 
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY, 
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ, 
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model, 
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH, 
    iou_thresh=NMS_IOU_THRESH
)